In [4]:
#!pip install img2vec_pytorch

### below is the dataset link i have used in this notebook

https://data.mendeley.com/datasets/4drtyfjtfy/1

In [5]:
import os
import pickle

from img2vec_pytorch import Img2Vec
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [6]:
!unzip '/content/weather_dataset.zip' -d '/content/weather_dataset'

Archive:  /content/weather_dataset.zip
  inflating: /content/weather_dataset/dataset2.zip  


In [8]:
!unzip '/content/weather_dataset/dataset2.zip' -d '/content/dataset'

Archive:  /content/weather_dataset/dataset2.zip
   creating: /content/dataset/dataset2/
  inflating: /content/dataset/dataset2/cloudy1.jpg  
  inflating: /content/dataset/dataset2/cloudy10.jpg  
  inflating: /content/dataset/dataset2/cloudy100.jpg  
  inflating: /content/dataset/dataset2/cloudy101.jpg  
  inflating: /content/dataset/dataset2/cloudy102.jpg  
  inflating: /content/dataset/dataset2/cloudy103.jpg  
  inflating: /content/dataset/dataset2/cloudy104.jpg  
  inflating: /content/dataset/dataset2/cloudy105.jpg  
  inflating: /content/dataset/dataset2/cloudy106.jpg  
  inflating: /content/dataset/dataset2/cloudy107.jpg  
  inflating: /content/dataset/dataset2/cloudy108.jpg  
  inflating: /content/dataset/dataset2/cloudy109.jpg  
  inflating: /content/dataset/dataset2/cloudy11.jpg  
  inflating: /content/dataset/dataset2/cloudy110.jpg  
  inflating: /content/dataset/dataset2/cloudy111.jpg  
  inflating: /content/dataset/dataset2/cloudy112.jpg  
  inflating: /content/dataset/datase

In [11]:
data_dir = '/content/dataset/dataset2'

In [12]:
img2vec = Img2Vec()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 118MB/s]


In [9]:
import os
import re
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split


def build_dataset(image_dir, img2vec, test_size=0.2, random_state=42):
    features = []
    labels = []

    for filename in os.listdir(image_dir):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):

            # Extract label: alphabetic prefix (cloudy100.jpg → cloudy)
            match = re.match(r"([a-zA-Z]+)", filename)
            if match is None:
                continue

            label = match.group(1)

            img_path = os.path.join(image_dir, filename)
            img = Image.open(img_path).convert("RGB")

            # Img2Vec feature extraction
            vec = img2vec.get_vec(img)

            features.append(vec)
            labels.append(label)

    features = np.array(features)
    labels = np.array(labels)

    # Train–Validation split (class-balanced)
    X_train, X_val, y_train, y_val = train_test_split(
        features,
        labels,
        test_size=test_size,
        stratify=labels,
        random_state=random_state
    )

    data = {
        "training_data": X_train,
        "training_labels": y_train,
        "validation_data": X_val,
        "validation_labels": y_val
    }

    return data

In [13]:
data = build_dataset(data_dir,img2vec=img2vec)

In [14]:
print(data)

{'training_data': array([[8.8185531e-01, 1.8535821e+00, 8.5386235e-01, ..., 1.3666356e+00,
        1.6800272e+00, 2.4572641e-01],
       [3.1773725e-01, 1.7649125e-01, 1.6110900e-01, ..., 1.7766718e+00,
        1.9370198e-01, 8.6150700e-01],
       [5.5781573e-02, 0.0000000e+00, 1.0831908e-01, ..., 1.1450278e-01,
        5.7064939e+00, 2.4171709e-01],
       ...,
       [4.1248283e-01, 1.0243261e+00, 8.4578192e-01, ..., 9.2202330e-01,
        2.7801576e+00, 1.1684309e-01],
       [3.1441948e+00, 2.4044991e-03, 6.6198632e-03, ..., 3.4244056e+00,
        1.8149326e+00, 1.4071146e-02],
       [8.4410316e-01, 1.9134642e+00, 4.9867499e-01, ..., 1.5302811e+00,
        8.9201915e-01, 1.2213644e-01]], dtype=float32), 'training_labels': array(['cloudy', 'rain', 'shine', 'cloudy', 'shine', 'rain', 'shine',
       'rain', 'cloudy', 'sunrise', 'cloudy', 'cloudy', 'rain', 'sunrise',
       'cloudy', 'shine', 'rain', 'sunrise', 'cloudy', 'shine', 'cloudy',
       'sunrise', 'rain', 'shine', 'cloudy'

In [15]:
model = RandomForestClassifier(random_state=0)
model.fit(data['training_data'], data['training_labels'])

RandomForestClassifier(random_state=0)

In [16]:
# test performance
y_pred = model.predict(data['validation_data'])
score = accuracy_score(y_pred, data['validation_labels'])
print(score)

0.96


In [17]:
# save the model
with open('./model.p', 'wb') as f:
    pickle.dump(model, f)
    f.close()

In [19]:
import pickle

from img2vec_pytorch import Img2Vec
from PIL import Image


with open('./model.p', 'rb') as f:
    model = pickle.load(f)

img2vec = Img2Vec()

image_path = '/content/dataset/dataset2/rain20.jpg'

img = Image.open(image_path)

features = img2vec.get_vec(img)

pred = model.predict([features])

print(pred)

['rain']
